# Os And Sys

> Generated from [`03-Os-And-Sys.py`](03-Os-And-Sys.py) by `tools/build_notebooks.py` - edit the `.py` (or its `.md` notes), not this notebook. Changes made here are overwritten on the next build.

## Quick notes

### What this covers
`os` (talk to the operating system: files, directories, env vars, processes) and `sys` (talk to the Python interpreter itself: args, path, exit codes), plus `subprocess` and `platform` for running commands and detecting the OS.

### Key points
- **`os.getcwd()`**, **`os.listdir(path)`** — current directory, its contents.
- **`os.makedirs(path, exist_ok=True)`** / **`os.rmdir(path)`** — create/remove directories.
- **`os.path`** — string-based path tools: `.join()`, `.basename()`, `.dirname()`, `.splitext()`, `.abspath()`, `.isfile()`/`.isdir()`.
- **`os.environ`** — dict-like access to environment variables; `os.environ["X"] = "y"` sets one for the current process.
- **`os.getenv(name, default)`** vs **`os.environ[name]`** — `getenv` returns a default safely, `environ[...]` raises `KeyError` if missing.
- **`os.walk(path)`** — recursively yields `(root, dirs, files)` for a directory tree.
- **`sys.argv`** — list of command-line arguments; `argv[0]` is the script name itself.
- **`sys.path`** — list of directories Python searches for imports.
- **`sys.version`**, **`sys.platform`** — interpreter version and OS platform string.
- **`sys.exit(code)`** — stops the script; `0` = success, non-zero = failure (checked by shells/CI).
- **`subprocess.run(["cmd", "arg"], capture_output=True, text=True)`** — run an external command; `.stdout`, `.returncode`.
- **`platform`** module — `.system()` ('Darwin'/'Linux'/'Windows'), `.release()`, `.machine()`, `.python_implementation()`.

### Gotchas (the part worth revising twice)
- **Always pass `subprocess.run()` a list of args, not a single string with `shell=True`** — a list avoids the argument being interpreted by a shell, which is how shell-injection bugs happen.
- `os.path` works with plain strings; `pathlib` (see file 05) is the modern object-oriented equivalent — prefer `pathlib` for new code.
- `os.getenv("X", default)` never raises; `os.environ["X"]` raises `KeyError` if unset — pick based on whether a missing var is actually an error.
- `sys.exit(code)` inside a subprocess only ends that subprocess — the convention `0=success` is what CI/shells check via `returncode`.

### When it matters
File/directory manipulation, reading config from environment variables, writing CLI scripts that need exit codes, or shelling out to another program safely.

### One-line recap example
```python
import subprocess
result = subprocess.run(["echo", "hi"], capture_output=True, text=True)
print(result.stdout.strip(), result.returncode)   # list of args, never shell=True
```

In [ ]:
import os
import sys

## 01. os and sys Modules - Introduction

- os: interact with the operating system (files, directories, env vars).
- sys: interact with the Python interpreter itself (args, path, exit).

## 02. Current Working Directory

In [ ]:
print("cwd:", os.getcwd())

## 03. Listing Directory Contents

In [ ]:
print(os.listdir("."))

## 04. Creating and Removing Directories

In [ ]:
os.makedirs("temp_demo_dir", exist_ok=True)
print("created temp_demo_dir, exists:", os.path.exists("temp_demo_dir"))
os.rmdir("temp_demo_dir")
print("removed temp_demo_dir, exists:", os.path.exists("temp_demo_dir"))

## 05. os.path - Path Manipulation

In [ ]:
sample_path = os.path.join("folder", "subfolder", "file.txt")
print("\n# 05. os.path")
print("joined path:", sample_path)
print("basename:", os.path.basename(sample_path))
print("dirname:", os.path.dirname(sample_path))
print("splitext:", os.path.splitext(sample_path))
print("absolute path:", os.path.abspath("."))
print("is file '.':", os.path.isfile("."))
print("is dir '.':", os.path.isdir("."))

## 06. Environment Variables

In [ ]:
print("HOME (or None):", os.environ.get("HOME"))
os.environ["MY_CUSTOM_VAR"] = "hello"
print("custom var:", os.environ.get("MY_CUSTOM_VAR"))

## 07. Walking a Directory Tree

In [ ]:
count = 0
for root, dirs, files in os.walk("."):
    count += 1
    if count > 2:  # keep demo output short
        break
    print(f"root={root}, dirs={dirs[:3]}, files={files[:3]}")

## 08. sys.argv - Command-Line Arguments

In [ ]:
print("script args:", sys.argv)  # sys.argv[0] is the script name itself

## 09. sys.path - Module Search Path

In [ ]:
print("first entry:", sys.path[0])
print("total entries:", len(sys.path))

## 10. sys.version and sys.platform

In [ ]:
print("Python version:", sys.version.split()[0])
print("platform:", sys.platform)

## 11. sys.exit() - Exiting a Script (mentioned, not called here)

## 12. os.path vs pathlib - Cross-Reference

## 13. subprocess.run() - Running Shell Commands Safely

- Pass the command as a LIST (not a single string with shell=True) so
  arguments aren't interpreted by a shell - avoids shell-injection risk.

In [ ]:
import subprocess

result = subprocess.run(["echo", "hello"], capture_output=True, text=True)
print("\n# 13. subprocess.run()")
print("stdout:", result.stdout.strip())
print("return code:", result.returncode)
print("succeeded:", result.returncode == 0)

## 14. os.getenv() vs os.environ[...] - Default vs KeyError

In [ ]:
print("getenv with default (safe):", os.getenv("DOES_NOT_EXIST", "fallback_value"))
try:
    os.environ["DOES_NOT_EXIST"]  # raises KeyError if the var isn't set
except KeyError as e:
    print("os.environ[...] raised KeyError:", e)

## 15. sys.exit() with Exit Codes (in a subprocess, so this file keeps running)

- Convention: 0 = success, non-zero = failure. Shells/CI pipelines check
  this code to decide whether a script "passed".

In [ ]:
child = subprocess.run(
    [sys.executable, "-c", "import sys; sys.exit(3)"]
)
print("child script exited with code:", child.returncode)

## 16. platform Module - OS Detection

In [ ]:
import platform

print("\n# 16. platform Module")
print("system:", platform.system())      # 'Darwin', 'Linux', 'Windows'
print("release:", platform.release())
print("machine:", platform.machine())    # e.g. 'arm64', 'x86_64'
print("python implementation:", platform.python_implementation())